In [0]:
-- Hospital Silver
-- Initial ID: 7,000,001
-- Run after successful Bronze ingestion.
-- Requires one concurrent writer.

CREATE SCHEMA IF NOT EXISTS pondolens.dev_silver;

-- Validate crosswalk uniqueness.
SELECT assert_true(
    COUNT(*) = 0,
    'Duplicate region aliases in shared_region_crosswalk.'
)
FROM (
    SELECT UPPER(TRIM(source_region_name))
    FROM pondolens.dev_silver.shared_region_crosswalk
    GROUP BY UPPER(TRIM(source_region_name))
    HAVING COUNT(*) > 1
);

-- Preserve invalid records in staging so validation can reject them.
CREATE OR REPLACE TEMP VIEW hospital_silver_stage AS
SELECT
    TRIM(h.year) AS raw_year,
    TRIM(h.regulated_hospital_count) AS raw_count,

    TRY_CAST(TRIM(h.year) AS INT) AS year,
    x.region_code,
    x.region_name,

    CASE UPPER(TRIM(h.hospital_type))
        WHEN 'GOVERNMENT' THEN 'Government'
        WHEN 'PRIVATE' THEN 'Private'
        WHEN 'TOTAL' THEN 'Total'
        ELSE TRIM(h.hospital_type)
    END AS hospital_type,

    TRY_CAST(TRIM(h.regulated_hospital_count) AS INT)
        AS regulated_hospital_count,

    LOWER(TRIM(h.unit)) AS unit

FROM pondolens.dev_bronze.hospital AS h

LEFT JOIN pondolens.dev_silver.shared_region_crosswalk AS x
    ON UPPER(TRIM(h.region_name))
        = UPPER(TRIM(x.source_region_name));

-- Block empty, invalid, or unmapped input.
SELECT assert_true(
    COUNT(*) > 0
    AND COUNT_IF(
        raw_year IS NULL
        OR NOT (raw_year RLIKE '^[0-9]{4}$')
        OR year IS NULL
        OR year NOT BETWEEN 1900 AND 9999

        OR NULLIF(TRIM(region_code), '') IS NULL
        OR NULLIF(TRIM(region_name), '') IS NULL

        OR hospital_type IS NULL
        OR hospital_type NOT IN ('Government', 'Private', 'Total')

        OR raw_count IS NULL
        OR NOT (raw_count RLIKE '^[0-9]+$')
        OR regulated_hospital_count IS NULL
        OR regulated_hospital_count < 0

        OR unit IS NULL
        OR unit <> 'hospitals'
    ) = 0,
    'Hospital load blocked: empty source, invalid values, or unmapped regions.'
)
FROM hospital_silver_stage;

SELECT assert_true(
    COUNT(*) = 0,
    'Duplicate year, region, and hospital type in source.'
)
FROM (
    SELECT
        year,
        region_code,
        hospital_type
    FROM hospital_silver_stage
    GROUP BY year, region_code, hospital_type
    HAVING COUNT(*) > 1
);

-- Create the target on the first run.
CREATE TABLE IF NOT EXISTS pondolens.dev_silver.hospital (
    hospital_id BIGINT,
    year INT,
    region_code STRING,
    region_name STRING,
    hospital_type STRING,
    regulated_hospital_count INT,
    unit STRING,
    processed_at TIMESTAMP
)
USING DELTA;

-- Remove provenance columns from the earlier table version.
-- Bronze retains the original source fields.
ALTER TABLE pondolens.dev_silver.hospital
SET TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name'
);

DECLARE OR REPLACE VARIABLE hospital_cleanup_sql STRING;

SET VAR hospital_cleanup_sql = (
    SELECT CASE
        WHEN COUNT(*) = 0 THEN
            'SELECT assert_true(true)'
        ELSE CONCAT(
            'ALTER TABLE pondolens.dev_silver.hospital DROP COLUMNS (',
            CONCAT_WS(
                ', ',
                SORT_ARRAY(
                    COLLECT_LIST(CONCAT('`', column_name, '`'))
                )
            ),
            ')'
        )
    END
    FROM pondolens.information_schema.columns
    WHERE table_catalog = 'pondolens'
      AND table_schema = 'dev_silver'
      AND table_name = 'hospital'
      AND column_name IN (
          'source_agency',
          'source_publication',
          'source_table',
          'source_page',
          'source_url',
          'coverage_note'
      )
);

EXECUTE IMMEDIATE hospital_cleanup_sql;

-- Validate existing target keys and IDs.
SELECT assert_true(
    COUNT_IF(
        hospital_id IS NULL
        OR hospital_id <= 7000000
        OR year IS NULL
        OR NULLIF(TRIM(region_code), '') IS NULL
        OR NULLIF(TRIM(hospital_type), '') IS NULL
    ) = 0,
    'Invalid existing hospital ID or natural key.'
)
FROM pondolens.dev_silver.hospital;

SELECT assert_true(
    COUNT(*) = 0,
    'Duplicate natural keys in hospital target.'
)
FROM (
    SELECT
        year,
        region_code,
        hospital_type
    FROM pondolens.dev_silver.hospital
    GROUP BY year, region_code, hospital_type
    HAVING COUNT(*) > 1
);

SELECT assert_true(
    COUNT(*) = 0,
    'Duplicate hospital_id values in target.'
)
FROM (
    SELECT hospital_id
    FROM pondolens.dev_silver.hospital
    GROUP BY hospital_id
    HAVING COUNT(*) > 1
);

-- Preserve IDs for matched keys.
-- Allocate new IDs only to new keys.
MERGE INTO pondolens.dev_silver.hospital AS target
USING (
    WITH maximum_id AS (
        SELECT
            COALESCE(
                CAST(MAX(hospital_id) AS BIGINT),
                CAST(7000000 AS BIGINT)
            ) AS last_id
        FROM pondolens.dev_silver.hospital
    ),
    matched AS (
        SELECT
            s.year,
            s.region_code,
            s.region_name,
            s.hospital_type,
            s.regulated_hospital_count,
            s.unit,
            t.hospital_id AS existing_id
        FROM hospital_silver_stage AS s
        LEFT JOIN pondolens.dev_silver.hospital AS t
            ON  t.year = s.year
            AND t.region_code = s.region_code
            AND t.hospital_type = s.hospital_type
    )
    SELECT
        COALESCE(
            CAST(m.existing_id AS BIGINT),
            maximum_id.last_id + ROW_NUMBER() OVER (
                PARTITION BY m.existing_id IS NULL
                ORDER BY
                    m.year,
                    m.region_code,
                    m.hospital_type
            )
        ) AS hospital_id,
        m.year,
        m.region_code,
        m.region_name,
        m.hospital_type,
        m.regulated_hospital_count,
        m.unit,
        CURRENT_TIMESTAMP() AS processed_at
    FROM matched AS m
    CROSS JOIN maximum_id
) AS source

ON  target.year = source.year
AND target.region_code = source.region_code
AND target.hospital_type = source.hospital_type

WHEN MATCHED AND (
    NOT (target.region_name <=> source.region_name)
    OR NOT (
        target.regulated_hospital_count
            <=> source.regulated_hospital_count
    )
    OR NOT (target.unit <=> source.unit)
    OR target.processed_at IS NULL
)
THEN UPDATE SET
    target.region_name = source.region_name,
    target.regulated_hospital_count = source.regulated_hospital_count,
    target.unit = source.unit,
    target.processed_at = source.processed_at

WHEN NOT MATCHED THEN INSERT (
    hospital_id,
    year,
    region_code,
    region_name,
    hospital_type,
    regulated_hospital_count,
    unit,
    processed_at
)
VALUES (
    source.hospital_id,
    source.year,
    source.region_code,
    source.region_name,
    source.hospital_type,
    source.regulated_hospital_count,
    source.unit,
    source.processed_at
);